# T08 — AR Model (AutoRegressive) — Book: CH05

**Methodology**: Marco Peixeiro, *Time Series Forecasting in Python*, Chapter 5.

### Book-mandated steps:
1. ADF stationarity test on health_index (level + first difference)
2. ACF and PACF plots to visually determine lag order p
3.  — select p by lowest AIC via SARIMAX(order=(p,0,0))
4. Fit best model → Ljung-Box residual test
5.  — walk-forward validation on representative engine
6. Full-dataset prediction → evaluate with RMSE + NASA score

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from functools import partial
import sklearn

# Book imports — exactly as CH05 uses them
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller

from src.models.classical import (
    build_pca_health_index, compute_failure_threshold,
    run_stationarity_report, plot_acf_pacf_multi, smooth_series,
    select_best_ar_order, _get_representative_engine,
    check_residuals, rolling_forecast_engine,
    predict_rul_ar, predict_dataset, RUL_CAP,
)
from src.evaluation.metrics import evaluate

PROC_DIR    = ROOT / "data" / "processed"
SENSOR_COLS = [f"s{i}" for i in [2,3,4,7,8,9,11,12,13,14,15,17,20,21]]

## 1. Load data and build health_index

In [ ]:
train = pd.read_csv(PROC_DIR / "train_features.csv")
test  = pd.read_csv(PROC_DIR / "test_features.csv")
train, test = build_pca_health_index(train, test, SENSOR_COLS)

THRESHOLD = compute_failure_threshold(train, end_of_life_rul=5, quantile=0.5)
print(f"Failure threshold: {THRESHOLD:.4f}")
print(f"hi_min={train.health_index.min():.3f} | hi_mean={train.health_index.mean():.3f}")

for q in [0.1, 0.2, 0.3, 0.5]:
    t = compute_failure_threshold(train, end_of_life_rul=5, quantile=q)
    hi_min = train["health_index"].min()
    reachable = "✓" if t > hi_min else "✗ below hi_min — forecast will never cross"
    print(f"  quantile={q} → threshold={t:.3f}  hi_min={hi_min:.3f}  {reachable}")

In [ ]:
from src.models.classical import fit_rul_from_health_index
fit_rul_from_health_index(train)

In [ ]:
# In any notebook, after build_pca_health_index
# This tells you EXACTLY what threshold value will work

hi_min  = train["health_index"].min()
hi_max  = train["health_index"].max()
hi_mean = train["health_index"].mean()
print(f"health_index range: [{hi_min:.3f}, {hi_max:.3f}]  mean={hi_mean:.3f}")

# Check what threshold each quantile gives
print("\nThreshold candidates:")
for q in [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]:
    t = compute_failure_threshold(train, end_of_life_rul=5, quantile=q)
    # Count test engines whose health_index ever reaches this threshold
    reachable = 0
    for _, g in test.groupby("engine_id"):
        if g["health_index"].max() >= t:
            reachable += 1
    pct = 100 * reachable / test["engine_id"].nunique()
    print(f"  q={q:.2f} → threshold={t:.3f}  |  {reachable}/{test['engine_id'].nunique()} "
          f"test engines reach it ({pct:.0f}%)")

In [ ]:
# Sanity check: health_index should increase as RUL decreases
from sklearn.metrics import r2_score
r2 = r2_score(-train["RUL"].values, train["health_index"].values)
print(f"health_index vs RUL R2: {r2:.3f}")
# If R2 < 0.1 → health_index is not tracking degradation → PCA sign flip failed
# If R2 > 0.3 → good signal

# Plot a few engines to visually confirm health_index rises over time
fig, axes = plt.subplots(1, 3, figsize=(14, 3))
for ax, eid in zip(axes, train["engine_id"].unique()[:248]):
    g = train[train["engine_id"] == eid].sort_values("cycle")
    ax.plot(g["cycle"], g["health_index"])
    ax.set_title(f"Engine {eid}")
    ax.set_xlabel("cycle"); ax.set_ylabel("health_index")
plt.tight_layout(); plt.show()
# Should show rising trend — if flat or noisy → PCA is not capturing degradation

In [ ]:
last_rul_per_engine = test.groupby("engine_id")["RUL"].last()
print("True RUL distribution of test engines:")
print(last_rul_per_engine.describe())
print(f"\nEngines with true RUL > 80:  {(last_rul_per_engine > 80).sum()}")
print(f"Engines with true RUL > 100: {(last_rul_per_engine > 100).sum()}")

## 2. Stationarity check — ADF test (CH03 methodology)

Book rule: run ADF at level + first difference. If level p > 0.05 and diff-1 p < 0.05 → d=1.
Here we check a stratified sample across ALL 4 FD subsets, not just 6 engines from FD001.

In [ ]:
stationarity_df = run_stationarity_report(train, n_engines=15)
MODAL_D = int(stationarity_df["recommended_d"].mode()[0])
print(f"\nNew MODAL_D = {MODAL_D}")

In [ ]:
# ── After running stationarity_df = run_stationarity_report(...) ──────

# 1. See d distribution across all engines
d_counts = stationarity_df["recommended_d"].value_counts().sort_index()
print("\nd=0 (already stationary)   :", d_counts.get(0, 0), "engines")
print("d=1 (1 difference needed)  :", d_counts.get(1, 0), "engines")
print("d=2 (2 differences needed) :", d_counts.get(2, 0), "engines")

# 2. Extract the modal d — this is what you pass to ARIMA
MODAL_D = int(stationarity_df["recommended_d"].mode()[0])
print(f"\n→ Use d = {MODAL_D} for AR")

# 3. Visual: compare raw vs differenced for one engine
raw  = train.sort_values("cycle").health_index.values
smth = smooth_series(raw, window=5)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))

axes[0].plot(smth)
axes[0].set_title("health_index — RAW (d=0)")
axes[0].set_xlabel("cycle")

diff1 = np.diff(smth, n=1)
axes[1].plot(diff1, color="orange")
axes[1].axhline(0, color="red", ls="--")
axes[1].set_title("health_index — after diff-1 (d=1)")
axes[1].set_xlabel("cycle")

plt.tight_layout()
plt.show()
# Raw: should show a clear downward trend (non-stationary)
# After diff-1: should hover around 0 with no trend (stationary)

## 3. ACF and PACF plots (CH05 methodology)

Book rule: PACF cuts off at lag p → AR order.
Plot on smoothed health_index (smoothing is applied before fitting in production too).

In [ ]:
# Pick one representative engine from FD001 to show ACF/PACF
raw   = train.sort_values("cycle").health_index.values
smth  = smooth_series(raw, window=5)

print(f" length: {len(smth)} cycles")
plot_acf_pacf_multi(train, d=MODAL_D, n_engines=2, lags=20)
print("Reading: PACF cuts off at lag p → candidate AR(p)")

## 4. Optimize AR order by AIC (CH05/CH06 pattern)

Book rule: run optimize function, sort by AIC ascending, pick lowest.

In [ ]:
BEST_P = select_best_ar_order(train, d=MODAL_D, n_engines=5)

## 5. Fit best AR model and check residuals (Ljung-Box)

Book rule (CH06/CH07): always run Ljung-Box after fitting. All p-values > 0.05 = white-noise residuals = adequate model.

In [ ]:
# Ljung-Box on longest engine — most stable for diagnostics
rep_eid, rep_smth = _get_representative_engine(train)
rep_diff = np.diff(rep_smth, n=MODAL_D)

model_fit = SARIMAX(rep_diff, order=(BEST_P, 0, 0), simple_differencing=False).fit(disp=False)
print(model_fit.summary())
lb_result = check_residuals(model_fit.resid, model_name=f"AR({BEST_P})")

## 6. Rolling forecast on representative engine (CH05 pattern)

Book rule: walk-forward validation — refit at each window step, predict out-of-sample.

In [ ]:
TRAIN_LEN = int(len(rep_diff) * 0.7)   # ← rep_diff, not rep_smth
WINDOW    = 1

pred_ar    = rolling_forecast_engine(
    series    = rep_diff,              
    train_len = TRAIN_LEN,
    order     = (BEST_P, 0, 0),       # d=0 is now valid — series is already stationary
    window    = WINDOW,
)
actual_val = rep_diff[TRAIN_LEN: TRAIN_LEN + len(pred_ar)]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rep_diff, color="steelblue", label="health_index (diff-2)")
ax.plot(range(TRAIN_LEN, TRAIN_LEN + len(pred_ar)), pred_ar,
        color="darkorange", ls="--", label=f"AR({BEST_P}) rolling forecast")
ax.axvline(TRAIN_LEN, color="gray", ls=":", label="train/val split")
ax.axhline(0, color="red", ls="--", label="zero line")   # ← threshold meaningless on diff series
ax.set_xlabel("cycle"); ax.set_ylabel("health_index (diff-2)")
ax.set_title(f"Rolling forecast — AR({BEST_P}) on diff-2 — engine {rep_eid}")
ax.legend(); plt.tight_layout(); plt.show()

rmse_roll = float(np.sqrt(np.mean((actual_val - pred_ar)**2)))
print(f"Rolling forecast RMSE on diff-2 health_index: {rmse_roll:.4f}")

## 7. Full test-set evaluation

In [ ]:
predict_fn = partial(predict_rul_ar, p=BEST_P, d=MODAL_D)
y_true, y_pred = predict_dataset(test, predict_fn, THRESHOLD)

print("=== AR Model — Test Results ===")
evaluate(y_true, y_pred, model_name=f"AR({BEST_P})")